In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Retina mode
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [ ]:
from latexify import latexify, format_axes

In [ ]:
latexify(columns=2, fig_height=4.5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.axes_grid1 import make_axes_locatable
import torch

def create_XYZ(f):
    # Generate data
    x = np.linspace(-5, 5, 100)
    y = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x, y)
    
    # Convert to PyTorch tensors
    X_torch = torch.from_numpy(X)
    Y_torch = torch.from_numpy(Y)

    # Evaluate the function
    Z = f(X_torch, Y_torch)
    return X, Y, Z, X_torch, Y_torch

def create_contour(X, Y, Z, ax2, alpha, scatter_pts=None, filled=True, levels=10, mark_levels=False):
    if filled:
        scatter_color='white'
        contour = ax2.contourf(X, Y, Z.detach().numpy(), levels=levels, cmap='magma', alpha=alpha)
    else:
        scatter_color='black'
        contour = ax2.contour(X, Y, Z.detach().numpy(), levels=levels, cmap='magma', alpha=alpha)
    if scatter_pts is not None:
        ax2.scatter(scatter_pts[0], scatter_pts[1], s=10, c=scatter_color)
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_title('Contour Plot')
    

    # Add a colorbar in between the subplots
    divider = make_axes_locatable(ax2)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(contour, cax=cax)
    return ax2, contour

def plot_surface_and_contour(f, function_name, uv = None, stride=4, alpha=1, scatter_pts=None, filled=True, levels=10):
    X, Y, Z, X_torch, Y_torch = create_XYZ(f)

    # Create the single figure with two subplots
    fig = plt.figure()

    # Plot the 3D surface on the first subplot
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(X, Y, Z.detach().numpy(), cmap='magma', edgecolor='none', alpha=alpha)  # Remove grid lines
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_zlabel('z')
    ax1.grid(False)
    ax1.xaxis.pane.fill = False
    ax1.yaxis.pane.fill = False
    ax1.zaxis.pane.fill = False
    ax1.view_init(elev=30, azim=30)
    ax1.set_title('Surface Plot')
    if scatter_pts is not None:
        ax1.scatter(scatter_pts[0], scatter_pts[1], f(scatter_pts[0], scatter_pts[1]), s=100, c='black')
    

    # Plot the contour plot on the second subplot
    ax2 = fig.add_subplot(122, aspect='equal')  # Set 1:1 aspect ratio
    
    ax2, contour = create_contour(X, Y, Z, ax2, alpha, scatter_pts, filled, levels)
    file_name = f"../figures/mml/contour-{function_name}.pdf"
    if uv is not None:
        u = uv[0](X_torch, Y_torch)
        v = uv[1](X_torch, Y_torch)
        # Quiver plot for gradient
        ax2.quiver(X[::stride, ::stride], Y[::stride, ::stride], u[::stride, ::stride].detach().numpy(),
                   v[::stride, ::stride].detach().numpy(), scale=140)
        # for c in contour, set alpha
        for c in contour.collections:
            c.set_alpha(0.5)
        
        
        file_name = f"../figures/mml/contour-{function_name}-with-gradient.pdf"
        

    
    # Save the figure
    plt.tight_layout(pad=1.0, w_pad=1.0)
    fig.savefig(file_name, bbox_inches="tight")

# Example usage:
# Define your function f(x, y) and its gradient g(x, y)
#f = lambda x, y: x**2 + y**2
#g = lambda x, y: (2*x, 2*y)
#plot_surface_and_contour(f, "x_squared_plus_y_squared", uv=(lambda x, y: 2*x, lambda x, y: 2*y))


In [ ]:
plot_surface_and_contour(lambda x, y: x**2 + y**2, "x_squared_plus_y_squared_quiver", 
                         uv=(lambda x, y: 2*x, lambda x, y: 2*y)
                         ,stride=5)

In [ ]:
plot_surface_and_contour(lambda x, y: x**2 + y**2, "x_squared_plus_y_squared")


In [ ]:
plot_surface_and_contour(lambda x, y: x**2, "x_squared")


In [ ]:
plot_surface_and_contour(lambda x, y: torch.abs(x) + torch.abs(y), "mod_x_plus_mod_y")


In [ ]:
plot_surface_and_contour(lambda x, y: (x**2) * y, "x_square_times_y")


In [ ]:
plot_surface_and_contour(lambda x, y: x * y, "x_times_y")


In [ ]:
def f(x, y):
    return (14 + 3*x**2 +14*y**2 -12*x - 28*y + 12*x*y)/3

x0, y0 = torch.tensor(4.0), torch.tensor(0.0)



In [ ]:
del_x, del_y = torch.func.grad(f, argnums=(0, 1))(x0, y0)
print(del_x, del_y)

In [ ]:
latexify(columns=2, fig_width=6.5, fig_height=3)
X, Y, Z, X_torch, Y_torch = create_XYZ(f)
levels = [0, 10, 25, 50, 100, 200, 300, 400]

xi = x0
yi = y0
alpha = 0.1
for i in range(50):
    fig, ax = plt.subplots(ncols=2)
    _, _ = create_contour(X, Y, Z, ax[0], alpha=0.8, scatter_pts=(xi, yi), filled=True, levels=levels)
    # Mark the minima with horizontal and vertical lines
    ax[0].axhline(y=yi, color='red', linestyle='--', alpha=0.7)
    ax[0].axvline(x=xi, color='red', linestyle='--', alpha=0.7)
    ax[0].scatter([0], [1], s=10, c='pink', marker='x', label='Minima')
    del_x, del_y = torch.func.grad(f, argnums=(0, 1))(xi, yi)
    xi = xi - alpha * del_x
    yi = yi - alpha * del_y
    print(xi, yi)
    
    # plot the line fit 
    x_line = np.linspace(0, 5, 100)
    y_line = xi + yi*x_line
    ax[1].plot(x_line, y_line, label='Line fit')
    ax[1].set_ylim(-1, 7)
    ax[1].plot(x_line, x_line, label='Actual line')
    ax[1].legend()
    
    plt.tight_layout()
    plt.savefig(f"../figures/mml/gradient-descent-{i}.pdf", bbox_inches="tight")





In [ ]:
levels = 10
def f1(x, y):
    return (2-x-2*y)**2

def f2(x, y):
    return (3-x-3*y)**2

def f3(x, y):
    return (1-x-y)**2

x0, y0 = torch.tensor(4.0), torch.tensor(0.0)


fig, ax = plt.subplots(ncols=3, figsize=(8, 3))
X, Y, Z, X_torch, Y_torch = create_XYZ(f1)
_, _ = create_contour(X, Y, Z, ax[0], alpha=0.8, scatter_pts=(x0, y0), filled=True, levels=levels)

X, Y, Z, X_torch, Y_torch = create_XYZ(f2)
_, _ = create_contour(X, Y, Z, ax[1], alpha=0.8, scatter_pts=(x0, y0), filled=True, levels=levels)

X, Y, Z, X_torch, Y_torch = create_XYZ(f3)
_, _ = create_contour(X, Y, Z, ax[2], alpha=0.8, scatter_pts=(x0, y0), filled=True, levels=levels)

fig.tight_layout()
plt.savefig(f"../figures/mml/gradient-descent-3-functions.pdf", bbox_inches="tight")

In [ ]:
X, Y, Z, X_torch, Y_torch = create_XYZ(f1)
alpha = 0.1


In [ ]:
del_x, del_y = torch.func.grad(f1, argnums=(0, 1))(x0, y0)
x1 = x0 - alpha * del_x
y1 = y0 - alpha * del_y

print(x1, y1)

In [ ]:
del_x, del_y = torch.func.grad(f2, argnums=(0, 1))(x1, y1)
x2 = x1 - alpha * del_x
y2 = y1 - alpha * del_y

print(x2, y2)

In [ ]:
del_x, del_y = torch.func.grad(f3, argnums=(0, 1))(x2, y2)
x3 = x2 - alpha * del_x
y3 = y2 - alpha * del_y

print(x3, y3)